# Exploration du catalogue Steam

Notebook de travail pour le projet NoSQL / Big Data.

L'idée de départ du sujet c'était MongoDB ou Neo4j. On a plutôt choisi **Qdrant** (base vectorielle) parce que le cas d'usage qu'on vise n'est pas vraiment du CRUD document classique : c'est de la **recherche sémantique** sur des descriptions de jeux.

**Problématique.** Un joueur ne connaît pas forcément le nom du jeu qu'il cherche. Il sait juste dire *« un jeu de cartes relaxant avec des puzzles »* ou *« un RPG sombre en monde ouvert »*. On veut pouvoir interroger le catalogue Steam avec ce genre de phrase, et ressortir des jeux proches.

**Données.**
- Source principale : un dump CSV du store Steam (~126k jeux).
- Source complémentaire (étape suivante) : Steam Web API, pour récupérer des infos plus à jour (détails / news) et dater les ingestions.

Ce notebook n'est pas "propre" volontairement : on garde les galères, les checks, et les allers-retours. L'ingest complet et l'appli viendront après.


## 1. Chargement du CSV

Premier souci, assez classique : le fichier n'est pas nickel. Le header a **deux colonnes collées** (`Discount` et `DLC count` → `DiscountDLC count`). Si on charge bêtement avec `pd.read_csv`, pandas décale toutes les colonnes à droite et on se retrouve avec n'importe quoi (des dates dans le nom, etc.).

On a perdu un moment là-dessus avant de s'en rendre compte.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

path = "../data/raw/games.csv"

columns = pd.read_csv(path, nrows=0).columns.tolist()
print("Nb colonnes lues dans le header :", len(columns))
print("Colonne bizarre :", [c for c in columns if "Discount" in c or "DLC" in c])

position = columns.index("DiscountDLC count")
columns[position:position + 1] = ["Discount", "DLC count"]

df = pd.read_csv(path, skiprows=1, names=columns)
df.shape


Petit check : après correction on doit avoir ~126k lignes et 40 colonnes. `AppID` doit être un entier, `Name` un vrai titre, `Release date` une date en texte.


In [ ]:
df.info()


In [ ]:
df.columns.tolist()


In [ ]:
df.head(3)


## 2. Qualité des données

Avant de nettoyer, on regarde un peu ce qui cloche : doublons, manquants, colonnes vides, types bizarres.


In [ ]:
print("Doublons complets :", df.duplicated().sum())
print("AppID uniques :", df["AppID"].nunique(), "/", len(df))
print("Noms manquants :", df["Name"].isna().sum())


In [ ]:
missing = (
    df.isna()
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
)
missing


### Ce qu'on retient

- **Pas de doublons** sur les lignes, et les `AppID` ont l'air uniques. Bon point.
- `Movies` est **100% vide**. Inutile.
- `Score rank` est quasi vide (environ 99.97%). Pareil.
- `Metacritic url`, `Reviews`, `Notes`, `Website` : beaucoup trop de NA pour s'en servir comme features principales.
- `Tags` manque pour ~1/3 des jeux. C'est embêtant pour la recherche, mais on a encore `Genres` / `About the game`.
- `About the game`, `Developers`, `Publishers`, `Categories`, `Genres` : autour de 7% de manquants. Souvent les mêmes lignes (jeux non listés, playtests, etc.).
- 1 jeu sans nom.

On va donc **jeter les colonnes mortes**, garder un sous-ensemble utile pour la recherche + quelques indicateurs (prix, notes, plateformes), et remplir les textes vides par `""` pour pouvoir concaténer.


In [ ]:
df.loc[df["Name"].str.contains("Playtest", case=False, na=False), ["AppID", "Name", "About the game", "Genres", "Tags"]].head()


## 3. Nettoyage

On copie le dataframe (histoire de pouvoir revenir au brut si besoin) puis on réduit.

Colonnes gardées :
- identifiant / nom
- texte (description, genres, tags, catégories, studios)
- un peu de contexte métier : prix, date, avis, metacritic, OS


In [ ]:
clean_df = df.copy()
clean_df = clean_df.drop_duplicates()

columns_to_keep = [
    "AppID",
    "Name",
    "About the game",
    "Genres",
    "Tags",
    "Developers",
    "Publishers",
    "Price",
    "Release date",
    "Positive",
    "Negative",
    "Recommendations",
    "Metacritic score",
    "Categories",
    "Windows",
    "Mac",
    "Linux",
]

clean_df = clean_df[columns_to_keep]
clean_df.shape


In [ ]:
clean_df = clean_df.dropna(subset=["AppID", "Name"])

text_columns = [
    "About the game",
    "Genres",
    "Tags",
    "Developers",
    "Publishers",
    "Categories",
]
clean_df[text_columns] = clean_df[text_columns].fillna("")

for col in ["Tags", "Genres", "Categories"]:
    clean_df[col] = clean_df[col].str.replace(",", ", ", regex=False)

clean_df.info()


### Features un peu plus propres

Rien de très malin : une année de sortie, un ratio d'avis positifs, un flag gratuit. Ça servira pour les graphes et plus tard pour filtrer dans Qdrant.


In [ ]:
clean_df["release_date"] = pd.to_datetime(clean_df["Release date"], errors="coerce")
clean_df["release_year"] = clean_df["release_date"].dt.year

n_reviews = clean_df["Positive"] + clean_df["Negative"]
clean_df["positive_ratio"] = np.where(
    n_reviews > 0,
    clean_df["Positive"] / n_reviews,
    np.nan,
)
clean_df["n_reviews"] = n_reviews
clean_df["is_free"] = clean_df["Price"] == 0

print("Dates non parsées :", clean_df["release_date"].isna().sum())
print("Jeux gratuits :", clean_df["is_free"].mean().round(3))
clean_df[["Name", "Release date", "release_year", "Price", "positive_ratio", "n_reviews"]].head()


Les playtests / outils n'ont souvent ni genre ni description. Pour la recherche sémantique ça pollue un peu (embedding quasi vide). On les met de côté pour les graphes "catalogue réel", mais on les garde dans `clean_df` au cas où.


In [ ]:
is_playtest = clean_df["Name"].str.contains("Playtest", case=False, na=False)
no_about = clean_df["About the game"].str.strip() == ""

print("Playtests :", is_playtest.sum())
print("Sans description :", no_about.sum())

games_df = clean_df[~is_playtest & ~no_about].copy()
print("Jeux gardés pour l'analyse / la recherche :", len(games_df), "/", len(clean_df))


## 4. Questions métier + viz

Avant d'envoyer quoi que ce soit dans Qdrant, on essaie de comprendre le catalogue. Questions qu'on s'est posées (à valider, mais ça nous paraît cohérent avec une appli de reco / recherche) :

1. Quels genres dominent Steam ?
2. Quelle est la répartition des prix ? Quelle part de free-to-play ?
3. Les jeux les mieux notés sont-ils plus chers ?
4. Le catalogue est-il vraiment multi-plateforme (Windows / Mac / Linux) ?
5. Comment le volume de sorties a évolué dans le temps ?

C'est le bloc ETL → query pandas → plot demandé dans le sujet. Les mêmes filtres (genre, prix, OS, année) serviront plus tard comme **payload** dans Qdrant.


### Q1 — Quels genres sont les plus représentés ?


In [ ]:
genres = (
    games_df["Genres"]
    .str.split(", ")
    .explode()
    .str.strip()
    .replace("", np.nan)
    .dropna()
)

top_genres = genres.value_counts().head(15)

plt.figure(figsize=(8, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, color="steelblue")
plt.title("Top 15 des genres (jeux avec description, hors playtests)")
plt.xlabel("Nombre de jeux")
plt.ylabel("")
plt.tight_layout()
plt.show()

top_genres


Pas de surprise : Indie / Casual / Action / Adventure écrasent le catalogue. C'est important pour la suite : une recherche sémantique qui se contenterait du genre "Indie" ne sert à rien, il faut vraiment passer par le texte de la description + les tags.


### Q2 — Prix : beaucoup de jeux à 0 € ?


In [ ]:
print(games_df["Price"].describe())
print("Part de jeux gratuits :", games_df["is_free"].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(games_df["Price"].clip(upper=60), bins=40, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution des prix (bornée à 60$)")
axes[0].set_xlabel("Prix")

games_df["is_free"].value_counts().rename({True: "Gratuit", False: "Payant"}).plot(
    kind="bar", ax=axes[1], color=["#4caf50", "#5c6bc0"], rot=0
)
axes[1].set_title("Gratuit vs payant")
axes[1].set_ylabel("Nombre de jeux")

plt.tight_layout()
plt.show()


Beaucoup de jeux très bas de gamme / gratuits. Un filtre `price` dans l'app aura du sens (budget étudiant vs AAA).


### Q3 — Est-ce que les jeux mieux notés sont plus chers ?


In [ ]:
rated = games_df[games_df["n_reviews"] >= 50].copy()
sample = rated.sample(n=min(8000, len(rated)), random_state=42)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=sample,
    x="Price",
    y="positive_ratio",
    alpha=0.25,
    s=18,
    edgecolor=None,
)
plt.xlim(0, 70)
plt.ylim(0, 1.02)
plt.title("Ratio d'avis positifs vs prix (jeux avec ≥ 50 avis, sample)")
plt.ylabel("Positive / (Positive + Negative)")
plt.tight_layout()
plt.show()

print("Corrélation prix / ratio :", rated[["Price", "positive_ratio"]].corr().iloc[0, 1].round(3))
rated.groupby(pd.cut(rated["Price"], bins=[-0.01, 0, 5, 15, 30, 70, 999]))["positive_ratio"].mean()


A priori pas de lien linéaire évident (le prix ne "fait" pas la note). En revanche le volume d'avis et le ratio seront utiles comme **signaux de popularité** pour reranker les résultats de recherche plus tard.


### Q4 — Windows / Mac / Linux


In [ ]:
os_share = games_df[["Windows", "Mac", "Linux"]].mean().mul(100).round(2)
print(os_share)

plt.figure(figsize=(5, 4))
sns.barplot(x=os_share.index, y=os_share.values, color="steelblue")
plt.title("Part des jeux par OS (%)")
plt.ylabel("% du catalogue")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

n_os = games_df[["Windows", "Mac", "Linux"]].sum(axis=1)
print("Nb d'OS supportés :")
print(n_os.value_counts().sort_index())


Windows est quasi universel. Mac / Linux sont minoritaires. Un filtre plateforme dans l'app n'est pas cosmétique : si on est sur Linux, une bonne partie du catalogue disparaît.


### Q5 — Volume de sorties dans le temps


In [ ]:
by_year = games_df["release_year"].value_counts().sort_index()
by_year = by_year[(by_year.index >= 2005) & (by_year.index <= 2026)]

plt.figure(figsize=(10, 4))
by_year.plot(kind="line", marker="o")
plt.title("Nombre de jeux sortis par année")
plt.xlabel("Année")
plt.ylabel("Nombre de jeux")
plt.tight_layout()
plt.show()

by_year.tail(10)


Explosion des sorties après 2014-2016 (Steam Direct / facilité de publication). Le dump contient donc beaucoup de petits jeux récents, ce qui justifie encore plus une recherche sémantique : le catalogue est trop gros pour scroller.


## 5. Texte qu'on va embedder

Qdrant stocke un vecteur + un payload. Le vecteur doit résumer *de quoi parle le jeu*. On concatène bêtement nom + description + genres/tags/catégories. C'est pas très subtil mais ça marche bien avec un modèle type MiniLM.

Plus tard on pourra enrichir avec des tags Steam plus frais via l'API.


In [ ]:
def build_embedding_text(row):
    parts = [
        str(row["Name"]),
        row["About the game"],
        "Genres: " + row["Genres"] if row["Genres"] else "",
        "Tags: " + row["Tags"] if row["Tags"] else "",
        "Categories: " + row["Categories"] if row["Categories"] else "",
    ]
    return ". ".join(p.strip() for p in parts if p and str(p).strip())

games_df["embedding_text"] = games_df.apply(build_embedding_text, axis=1)
clean_df["embedding_text"] = clean_df.apply(build_embedding_text, axis=1)

print(games_df["embedding_text"].iloc[1])
print("---")
print("Longueur moyenne (caractères) :", games_df["embedding_text"].str.len().mean().round(0))
print("Textes très courts (< 80 chars) :", (games_df["embedding_text"].str.len() < 80).sum())
games_df[["AppID", "Name", "embedding_text"]].head(3)


## 6. POC embeddings + Qdrant

On ne va **pas** indexer les 126k jeux depuis le notebook (trop long, et ça n'a rien à faire ici). L'idée c'est juste de vérifier que la chaîne marche :

`texte → embedding (384 dims) → collection Qdrant → query en langage naturel`

Prérequis : Qdrant lancé en local (`localhost:6333`). Le modèle `all-MiniLM-L6-v2` est petit, ça tourne sur CPU.

**Erreur qu'on a faite au début :** on avait mis `id=0, 1, 2, ...` au lieu de l'`AppID`. Du coup impossible de mettre à jour un jeu proprement. On utilise l'AppID maintenant.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

test_texts = games_df["embedding_text"].head(5).tolist()
test_embeddings = model.encode(test_texts)
test_embeddings.shape


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient("localhost", port=6333)
client.get_collections()


In [ ]:
TEST_COLLECTION = "steam_games_poc"

if client.collection_exists(TEST_COLLECTION):
    client.delete_collection(TEST_COLLECTION)

client.create_collection(
    collection_name=TEST_COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

sample_df = games_df.head(10).copy()
vectors = model.encode(sample_df["embedding_text"].tolist(), show_progress_bar=True)

points = []
for i, (_, row) in enumerate(sample_df.iterrows()):
    payload = {
        "app_id": int(row["AppID"]),
        "name": row["Name"],
        "genres": row["Genres"],
        "tags": row["Tags"],
        "categories": row["Categories"],
        "developers": row["Developers"],
        "publishers": row["Publishers"],
        "price": float(row["Price"]),
        "release_date": row["Release date"],
        "positive": int(row["Positive"]),
        "negative": int(row["Negative"]),
        "windows": bool(row["Windows"]),
        "mac": bool(row["Mac"]),
        "linux": bool(row["Linux"]),
    }
    points.append(
        PointStruct(
            id=int(row["AppID"]),
            vector=vectors[i].tolist(),
            payload=payload,
        )
    )

client.upsert(collection_name=TEST_COLLECTION, points=points)
client.count(TEST_COLLECTION)


In [ ]:
query = "A relaxing card game with puzzles"
query_vector = model.encode(query).tolist()

results = client.query_points(
    collection_name=TEST_COLLECTION,
    query=query_vector,
    limit=5,
)

for point in results.points:
    print(point.payload["name"])
    print("score:", round(point.score, 3), "| prix:", point.payload["price"], "| genres:", point.payload["genres"])
    print("-" * 50)


Avec seulement 10 jeux le ranking n'a pas beaucoup de sens (on cherche dans une poignée de titres). Mais ça valide le tuyau : on encode une phrase en anglais, Qdrant renvoie des payloads.

À faire ensuite (pas dans ce notebook) :
- ingest du catalogue complet, par batch
- indexes sur le payload (genre, prix, OS, année) pour filtrer
- scripts CRUD / snapshot
- 2e source Steam API + champ `ingested_at`
- petite webapp (Streamlit) par-dessus


## 7. Bilan de l'exploration

**Problèmes rencontrés**
- Header CSV cassé (`DiscountDLC count`) → colonnes décalées si on ne corrige pas.
- Colonnes 100% vides ou quasi (`Movies`, `Score rank`).
- Beaucoup de playtests / jeux sans description → embeddings pauvres.
- 1 nom manquant.
- Dates Steam pas toujours parsables (`errors="coerce"`).
- Premier test Qdrant avec des ids 0..n : mauvaise idée, on passe par `AppID`.

**Ce qui est prêt**
- Jeu nettoyé, texte d'embedding, quelques features (prix, ratio, année, OS).
- Questions métier + graphes.
- POC Qdrant (10 points).

**Limites**
- Dataset statique (dump). D'où l'intérêt de l'API Steam ensuite.
- Pas d'index payload / pas d'ingest massif ici.
- Modèle MiniLM en anglais : les requêtes FR marcheront moins bien. À voir si on reste en anglais dans l'UI ou si on change de modèle plus tard.
